In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import requests
from bs4 import BeautifulSoup
import re
from torch.utils.data import Dataset, DataLoader

## Dataset scraping and preparation

In [2]:
print("Scraping and preparing data")
url = "https://www.gutenberg.org/cache/epub/84/pg84-images.html"
response = requests.get(url)
soup = BeautifulSoup(response.content, 'html.parser')
raw_text = soup.get_text()
tokens = re.findall(r'\b\w+\b', raw_text.lower())

# Build Vocabulary
vocab = sorted(list(set(tokens)))
word_to_idx = {word: i for i, word in enumerate(vocab)}
idx_to_word = {i: word for i, word in enumerate(vocab)}
vocab_size = len(vocab)

# Define window_size variable
window_size = 100
seq_length = window_size - 1 # 99 tokens for input

inputs = []
targets = []

# Format into lists where inner lists contain 99 numbers
for i in range(len(tokens) - window_size + 1):
    seq_in = tokens[i : i + seq_length]
    seq_out = tokens[i + seq_length]

    # Input sequence and target are consecutive
    inputs.append([word_to_idx[w] for w in seq_in])
    targets.append(word_to_idx[seq_out])

class TextDataset(Dataset):
    def __init__(self, x_data, y_data):
        self.x = torch.tensor(x_data, dtype=torch.long)
        self.y = torch.tensor(y_data, dtype=torch.long)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

dataset = TextDataset(inputs, targets)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)


Scraping and preparing data


## Model architecture

### RNN model

In [3]:
class TextGenerationRNN(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super(TextGenerationRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        # Using ReLU to prevent vanishing gradients over the 99-token window
        self.rnn = nn.RNN(input_size=embed_size, hidden_size=hidden_size,
                          num_layers=1, nonlinearity='relu', batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        embedded = self.embedding(x)
        out, hidden = self.rnn(embedded)

        last_out = out[:, -1, :]
        logits = self.fc(last_out)
        return logits

### LSTM model

In [4]:
class TextGenerationLSTM(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super(TextGenerationLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(input_size=embed_size, hidden_size=hidden_size,
                            num_layers=1, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        embedded = self.embedding(x)
        out, (hidden, cell) = self.lstm(embedded)

        last_out = out[:, -1, :]
        logits = self.fc(last_out)
        return logits


### Model Training

In [7]:
def train_model(model, dataloader, epochs=15, learning_rate=0.001):
    model.train()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    print(f"--- Training {model.__class__.__name__} ---")
    for epoch in range(epochs):
        epoch_loss = 0
        for batch_inputs, batch_targets in dataloader:
            optimizer.zero_grad()

            outputs = model(batch_inputs)
            loss = criterion(outputs, batch_targets)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            epoch_loss += loss.item()

        # print loss for each epoch
        print(f"Epoch {epoch+1}/{epochs} | Loss: {epoch_loss/len(dataloader):.4f}")

    return model

In [8]:
print("Starting training...")

# Hyperparameters
embed_size = 64
hidden_size = 128
epochs = 15

# Initialize models
rnn_model = TextGenerationRNN(vocab_size, embed_size, hidden_size)
lstm_model = TextGenerationLSTM(vocab_size, embed_size, hidden_size)

# Train both models
rnn_model = train_model(rnn_model, dataloader, epochs=epochs, learning_rate=0.001)
print("\n")
lstm_model = train_model(lstm_model, dataloader, epochs=epochs, learning_rate=0.001)

Starting training...
--- Training TextGenerationRNN ---
Epoch 1/15 | Loss: 7.2529
Epoch 2/15 | Loss: 5.5926
Epoch 3/15 | Loss: 5.2050
Epoch 4/15 | Loss: 4.8594
Epoch 5/15 | Loss: 4.5108
Epoch 6/15 | Loss: 4.1609
Epoch 7/15 | Loss: 3.8380
Epoch 8/15 | Loss: 3.5565
Epoch 9/15 | Loss: 3.3211
Epoch 10/15 | Loss: 3.1176
Epoch 11/15 | Loss: 2.9429
Epoch 12/15 | Loss: 2.7916
Epoch 13/15 | Loss: 2.6571
Epoch 14/15 | Loss: 2.5377
Epoch 15/15 | Loss: 2.4279


--- Training TextGenerationLSTM ---
Epoch 1/15 | Loss: 6.4901
Epoch 2/15 | Loss: 5.7634
Epoch 3/15 | Loss: 5.3714
Epoch 4/15 | Loss: 5.0233
Epoch 5/15 | Loss: 4.7038
Epoch 6/15 | Loss: 4.4075
Epoch 7/15 | Loss: 4.1279
Epoch 8/15 | Loss: 3.8662
Epoch 9/15 | Loss: 3.6201
Epoch 10/15 | Loss: 3.3921
Epoch 11/15 | Loss: 3.1817
Epoch 12/15 | Loss: 2.9850
Epoch 13/15 | Loss: 2.8040
Epoch 14/15 | Loss: 2.6359
Epoch 15/15 | Loss: 2.4784


## Text Generation

In [9]:
def generate_text(model, seed_sentence, word_to_idx, idx_to_word, seq_length=99, num_words=100):
    model.eval()
    words = seed_sentence.lower().split()

    with torch.no_grad():
        for _ in range(num_words):
            # Take up to the last 99 words for the context window
            context_words = words[-seq_length:]

            # Convert to indices, using 0 for unknown words
            context_indices = [word_to_idx.get(w, 0) for w in context_words]

            # Pad sequence if the seed is shorter than 99 words (pads with 0 at the beginning)
            if len(context_indices) < seq_length:
                padding = [0] * (seq_length - len(context_indices))
                context_indices = padding + context_indices

            x_tensor = torch.tensor([context_indices], dtype=torch.long)

            prediction = model(x_tensor)
            predicted_idx = torch.argmax(prediction, dim=-1).item()
            predicted_word = idx_to_word[predicted_idx]

            words.append(predicted_word)

    return ' '.join(words)


## Comparison of RNN and LSTM

In [10]:
# Generate text using the exact same seed sentence
seed_sentence = "the monster looked at me and"

print("\n=== Generation Results ===")
print("Seed Sentence:", seed_sentence)

print("\n[Vanilla RNN Output]:")
rnn_text = generate_text(rnn_model, seed_sentence, word_to_idx, idx_to_word)
print(rnn_text)

print("\n[LSTM Output]:")
lstm_text = generate_text(lstm_model, seed_sentence, word_to_idx, idx_to_word)
print(lstm_text)


=== Generation Results ===
Seed Sentence: the monster looked at me and

[Vanilla RNN Output]:
the monster looked at me and then himself corrected and augmented them with sleep and the sun became heatless rain and twenty enemy but i am glad to me and i should have been imbued with her confessing and i remained with a serious anguish that i should have been so cruelly deprived me i was occupied in fixing the past time that i was carried back and the sun became heatless rain and twenty enemy but i am glad to me and i should have been imbued with her confessing and i remained with a serious anguish that i should have been so cruelly deprived

[LSTM Output]:
the monster looked at me and my father was to me with a mixture of affection and that consoles me as i was my own mind to the first time i felt as if i had been my adjuration to profound and partly to him he had been out of the same day but when i found that i had been the energy of my friend was still more deeply than the flight of th